In [77]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import cv2
from PIL import Image

# Visual Analytics & Virality Prediction

Every single second, thousands of images are uploaded to attention-driven platforms like Instagram and TikTok. While traditional digital marketing models heavily focus on text metadata—such as hashtags, captions, and posting time—modern recommendation algorithms prioritize a visual-first approach.This notebook explores a fundamental question: Can we predict if an image will trend based purely on its raw mathematical pixel properties?Instead of using heavy, black-box deep learning architectures, this project focuses on clear statistical inference. We treat every image as a raw 3D matrix tensor ($I \in \mathbb{R}^{M \times N \times 3}$) and extract three key visual indicators:

1. Color Saturation: The structural richness and vibrancy of the color palette.

2. Visual Clutter (Edge Density): The complexity and minimalist vs. chaotic nature of the composition.

3. Brightness Contrast: The variance of lighting and depth across the image matrix.

Using a part of the open-source AVA (Aesthetic Visual Analysis) dataset, I will mathematically map out the differences between high-engagement (viral) images and low-performing ones, ultimately building an interpretable statistical logistic regression model to calculate a post's probability of trending.

## Mathematical Formulation

### The Input Space (Image Tensors)

A digital image is not merely a graphic file; it is a multi-dimensional real-valued tensor:
$$I \in \mathbb{R}^{M \times N \times 3}$$

Where:
- $M$ represents the discrete row index (pixel height).
- $N$ represents the discrete column index (pixel width).
- $3$ represents the color channel depth mapped to the normalized RGB/HSV vector space.

### The Feature Mapping Function

Defining a vector-valued feature extraction function $f(I)$ that compresses the high-dimensional image tensor into a low-dimensional space containing the three target visual metrics:
$$f(I) = \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} \mu_{\text{(saturation)}} \\ \rho_{\text{(clutter)}} \\ \sigma_{\text{(contrast)}} \end{bmatrix}$$

This function maps our data from $\mathbb{R}^{M \times N \times 3} \rightarrow \mathbb{R}^3$, allowing feeding traditional statistical models without encountering a problem with the dimensionality (running out of computing power because there are too many data points).

### The Target Space (Engagement Optimization)

Defining the target domain as a discrete binary set:
$$Y \in \{0, 1\}$$
Where $Y = 1$ indicates a "viral" image (top-tier community aesthetic rating) and $Y = 0$ indicates a non-trending one. The ultimate objective is to approximate the true conditional probability distribution:
$$P(Y = 1 \mid X)$$

## Data Pipeline & Local Batch Processing

In [42]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATASET_DIR = r"C:\Users\ivona\Downloads\VP" 
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
METADATA_PATH = os.path.join(DATASET_DIR, "ground_truth_dataset.csv")

In [61]:
columns = [
    'image_id', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 
    'tag1', 'tag2', 'challenge_id'
]
df_raw = pd.read_csv(METADATA_PATH, sep=',')

In [64]:
# Calculate true continuous aesthetic means via vector dot-product
vote_columns = [f'vote_{i}' for i in range(1, 11)]
votes = df_raw[vote_columns].values
score_weights = np.arange(1, 11)
df_raw['mean_score'] = np.sum(votes * score_weights, axis=1) / np.sum(votes, axis=1)

In [65]:
# Establish binary target classification matrix threshold (>= 5.5 = Viral)
df_raw['is_viral'] = (df_raw['mean_score'] >= 5.5).astype(int)

In [68]:
# Map dynamic absolute paths for physical disk checks
df_raw['image_id_clean'] = df_raw['image_num'].astype(str)
df_raw['file_path'] = df_raw['image_id_clean'].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))

print(f"\n-> Raw catalog parsed: {len(df_raw):,} entries loaded.")


-> Raw catalog parsed: 255,508 entries loaded.


In [69]:
print("\nValidating physical image file records on disk...")
df_existing = df_raw[df_raw['file_path'].apply(os.path.exists)].copy()
print(f"-> Verified available assets: {len(df_existing):,} files present.")


Validating physical image file records on disk...
-> Verified available assets: 255,508 files present.


In [73]:
#Isolate a controlled balanced subset from the file list
sample_size_per_class = 500

flops = df_existing[df_existing['is_viral'] == 0].sample(n=sample_size_per_class, random_state=RANDOM_SEED)
virals = df_existing[df_existing['is_viral'] == 1].sample(n=sample_size_per_class, random_state=RANDOM_SEED)
df_balanced = pd.concat([flops, virals]).reset_index(drop=True)


df_train, df_test = train_test_split(
    df_balanced, 
    test_size = 0.20, 
    stratify = df_balanced['is_viral'], 
    random_state = RANDOM_SEED
)

print(f" -> Training Partition: {len(df_train)} images")
print(f" -> Testing Partition:  {len(df_test)} images")

 -> Training Partition: 800 images
 -> Testing Partition:  200 images


### Feature extraction & matrix pipeline

With the data governance rules established and operational cohorts split, initiate the pixel-level feature extraction pipeline. To pass the paths to the external module `src.image_features` to compute low-dimensional vectors from the raw multi-dimensional image tensors.

#### Extraction Methodology
To preserve strict system isolation, i extract structural and chromatic features from `df_train` and `df_test` as distinct operations. 

To capture:
1. **Saturation:** Calculated from the secondary channel of the HSV transformation matrix.
2. **Visual Clutter:** Structure pixel intensity density calculated via a Canny edge filter.
3. **Contrast:** Evaluated via the global standard deviation of the converted Grayscale luminance array.

In [79]:
# Define the image processing matrix math
def extract_visual_features(pil_image):
    # Convert PIL Image format to OpenCV standard matrix (RGB to BGR)
    img_rgb = np.array(pil_image.convert('RGB'))
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    
    # Convert matrix to hsv color space (Hue, Saturation, Value)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    avg_saturation = float(np.mean(hsv[:, :, 1])) # Index 1 is the Saturation matrix

    # Convert matrix to 2D grayscale and apply a Canny edge-detection filter
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    
    # Ratio of edge boundary pixels over total pixel space
    edge_density = float(np.sum(edges > 0) / (edges.shape[0] * edges.shape[1]))

    # Statistical standard deviation of the grayscale luminance matrix
    brightness_std = float(np.std(gray))
    
    return avg_saturation, edge_density, brightness_std

# The partition processing engine
def process_partition_features(df_partition, partition_name="Dataset"):
    print(f"\n--- Starting Feature Extraction: {partition_name} Cohort ---")
    
    saturations = []
    clutters = []
    contrasts = []
    total_records = len(df_partition)
    
    for idx, path in enumerate(df_partition['file_path']):
        try:
            with Image.open(path) as img:
                # Call the defined function above
                sat, clutter, contrast = extract_visual_features(img)
                saturations.append(sat)
                clutters.append(clutter)
                contrasts.append(contrast)
                
            # Track progress every 50 files for runtime transparency
            if (idx + 1) % 50 == 0 or (idx + 1) == total_records:
                print(f" -> Processed {idx + 1}/{total_records} images successfully.")
                
        except Exception as e:
            print(f" ! Exception at index {idx} for path [{path}]: {e}")
            saturations.append(0.0)
            clutters.append(0.0)
            contrasts.append(0.0)
            
    df_out = df_partition.copy()
    df_out['saturation'] = saturations
    df_out['clutter'] = clutters
    df_out['contrast'] = contrasts
    return df_out

# Execute the pipeline

# Run extraction independently on each split to prevent data leakage
df_train_extracted = process_partition_features(df_train, partition_name="Training")
df_test_extracted = process_partition_features(df_test, partition_name="Testing")

print("\n=== Extraction pipeline successful ===")
print("Training Sample Snapshot:")
print(df_train_extracted[['image_num', 'is_viral', 'saturation', 'clutter', 'contrast']].head())


--- Starting Feature Extraction: Training Cohort ---
 -> Processed 50/800 images successfully.
 -> Processed 100/800 images successfully.
 -> Processed 150/800 images successfully.
 -> Processed 200/800 images successfully.
 -> Processed 250/800 images successfully.
 -> Processed 300/800 images successfully.
 -> Processed 350/800 images successfully.
 -> Processed 400/800 images successfully.
 -> Processed 450/800 images successfully.
 -> Processed 500/800 images successfully.
 -> Processed 550/800 images successfully.
 -> Processed 600/800 images successfully.
 -> Processed 650/800 images successfully.
 -> Processed 700/800 images successfully.
 -> Processed 750/800 images successfully.
 -> Processed 800/800 images successfully.

--- Starting Feature Extraction: Testing Cohort ---
 -> Processed 50/200 images successfully.
 -> Processed 100/200 images successfully.
 -> Processed 150/200 images successfully.
 -> Processed 200/200 images successfully.

=== Extraction pipeline successful

### Feature-level data cleaning & validation

Before passing the extracted visual metrics into exploratory visualizations or statistical models, i'll perform defensive data cleaning on the independent partitions. 

By scaning both cohorts for null values and purge extraction failures (e.g., solid black frames or unparseable images that defaulted to `0.0` values during OpenCV matrix operations).

In [82]:
def clean_extracted_features(df_partition, partition_name="Dataset"):
    print(f"Cleaning Matrix: {partition_name}")
    print(f" -> Row count before cleaning: {len(df_partition)}")
    
    # Purge anomalies (images that returned zeroes across features)
    invalid_rows = df_partition[
        (df_partition['saturation'] == 0.0) | 
        (df_partition['clutter'] == 0.0) | 
        (df_partition['contrast'] == 0.0)
    ]
    print(f" -> Dropping {len(invalid_rows)} corrupted/edge-case rows.")
    df_clean = df_partition.drop(invalid_rows.index).copy()
    
    # Complete a null-value check
    df_clean = df_clean.dropna(subset=['saturation', 'clutter', 'contrast'])
    
    # Restrict columns to essential features and labels to save space
    columns_to_keep = ['image_num', 'mean_score', 'is_viral', 'saturation', 'clutter', 'contrast']
    df_final = df_clean[columns_to_keep].reset_index(drop=True)
    
    print(f"Final clean row count: {len(df_final)}\n")
    return df_final

# Run the cleaner independently across both splits to prevent data leakage
df_train_clean = clean_extracted_features(df_train_extracted, partition_name="TRAINING SET")
df_test_clean = clean_extracted_features(df_test_extracted, partition_name="TESTING SET")

print("Cleaned Training Data Sample:")
print(df_train_clean.head())

Cleaning Matrix: TRAINING SET
 -> Row count before cleaning: 800
 -> Dropping 85 corrupted/edge-case rows.
Final clean row count: 715

Cleaning Matrix: TESTING SET
 -> Row count before cleaning: 200
 -> Dropping 29 corrupted/edge-case rows.
Final clean row count: 171

Cleaned Training Data Sample:
   image_num  mean_score  is_viral  saturation   clutter   contrast
0     934744    6.251337         1  161.008962  0.055797  43.041668
1     836669    4.808219         0   75.089002  0.079823  50.090601
2     820679    6.730570         1  132.388450  0.066796  46.423987
3     693915    5.020408         0   47.562079  0.066928  79.258239
4     120626    5.660550         1   18.040605  0.112575  67.533989
